# nano Beatrix — the find screen's members: real arms on Alhena-F

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_find_members.ipynb)

**What trains**: the scale test's members verbatim on up to three cores (trained by `nano_find_cores.ipynb`): the replication core (the scale test's Alhena seed 1 trained again, series `scalerep`) and Alhena-F on seeds 1 and 2 (or a seed's 48,000-step retrain when the cores' notebook ran it). A core carries members only when its own end read answers at least .95 of its questions whole with the true address: a member's answer is the value the core retrieves, so on a core that does not retrieve it no member can answer.

| stage | members | where they write | budget |
|---|---|---|---|
| last site | 48 real arms (the arm library's RelayPatchwork, 16 slots x 8 through an aleph read of 16 anchors, born inert) + 48 toy members (the replication arm) | after the last block, before the final norm | 16,000 steps, each member 3 of its questions a step |
| mid site | the first 16 owners as real arms | after block 3 of 6 | 32,000 batch steps, one row a member |

Each member answers its own record's questions in its own letter code, trained ALONE with its own pure Adam on the frozen core (the true address in the training rows' streams). The read merges 16 / 32 / 48 of them under the gate w_i = sinh(u_i) / (1 + sum_j (cosh(u_j) - 1)) and grades PROTECTED, QUIET and EXACT per cell. On Alhena-F the read is the two-pass form: each request source puts its own address in the questions' streams and its own scores or kernels in the gate - THE CLAIM is HEADIF (the head at the asked key's index entry on the free-running row; its .5 decode in the stream, its scores in the gate) and HEADIFQ999 (the same through the absolute floor). The replication core is read in both forms (the scale test's, to compare its cells with the stored ones; and the two-pass form).

**The cells**: C1 pins the trainer and the arm library and holds the decision block; C2 builds the data, picks and fetches the cores, checks the compiled writes against eager at step 1 on this card, prices 200 steps of each trainer (a projection past 1.5x the plan stops for a re-plan) and probes the push path; C3 runs the cores in order, each core's folder shipped to [AbstractPhil/alephllm-nano-beatrix-training](https://huggingface.co/AbstractPhil/alephllm-nano-beatrix-training) under `bed/scale_members/`; C4 renders the table.

**Before C1**: the notebook's `HF_TOKEN` secret (write access to that repo) with notebook access on. A file named `STOP` in `/content/out` ends the session between cores; a restarted session resumes. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/rotary_scale.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.72" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| card',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 72), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the runner and the shipper read it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting set from the record, with its source beside it ============
OUT = '/content/out'
REPO = 'AbstractPhil/alephllm-nano-beatrix-training'  # the cores come from here; every finished core's members ship here (bed/scale_members/)
CORPUS_BYTES = 160e6                                   # the bed corpus (the members' training rows draw their filler from it)
TAIL = '_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50_ahdigit1'
REP_CORE = (f'alhena-both_s1_L24000{TAIL}_scalerep', 1, ('scale', 'find'))   # the replication core: both read forms
FIND_SEEDS = (1, 2)                                    # Alhena-F: a seed's 48,000-step retrain when it exists, else its 24,000-step core
MIN_ANSWERS = 0.95                                     # a core carries members only when its own end read answers >= .95 of its questions
                                                       # whole with the true address: a member's answer is the value the core retrieves
LAST_STEPS = 16000                                     # the scale test's member budget, verbatim
MID_STEPS = 32000                                      # batch steps at the mid site (16,000 own steps a member)
READ_BATCHES, CALIB_BATCHES = 64, 32                   # the cores' own read and calibration rows (8 rows a batch)
PRICE_STEPS = 200                                      # C2's price lines (each trainer on throwaway stacks)
PLAN_MIN = 18.8                                        # the planned training minutes of one core's members on this card (4.9 last + 13.9 mid)
STOP_FACTOR = 1.5
print('decision block:', 1 + len(FIND_SEEDS), 'cores |', LAST_STEPS, 'last-site steps |', MID_STEPS, 'mid-site batch steps | ships to',
      REPO if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the data, the cores, compiled = eager at step 1 on this card, the price lines, the push probe. No member trains.
import json, time
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
from geolip.nano_beatrix import bed_corpus, rotary_scale as rs
BED_DIR = f'{OUT}/bed'
if not os.path.exists(f'{BED_DIR}/bed_corpus.bin'):
    bed_corpus.build(BED_DIR, CORPUS_BYTES)
HELD = f'{BED_DIR}/heldout_fineweb_edu_head2048.jsonl'
print('corpus', os.path.getsize(f'{BED_DIR}/bed_corpus.bin') / 1e6, 'MB; held-out head', os.path.exists(HELD))
files = set(HfApi(token=HF_TOKEN or False).list_repo_files(REPO))
CAND = [REP_CORE]
for seed in FIND_SEEDS:                                # the registered retrain replaces its seed's 24,000-step core
    tags = [f'alhena-both_s{seed}_L{st}{TAIL}i1_find' for st in (48000, 24000)]
    have = [t for t in tags if f'bed/weights/{t}/model.safetensors' in files]
    assert have, f'no Alhena-F core of seed {seed} on the Hub: run nano_find_cores.ipynb first'
    CAND.append((have[0], seed, ('find',)))
CORES = []                                             # the core's own end read decides, from its record on the Hub
for tag, seed, forms in CAND:
    p = hf_hub_download(REPO, f'bed/reports/bed_{tag}.json', local_dir=f'{OUT}/hub', token=HF_TOKEN or False)
    ans = json.load(open(p, encoding='utf-8'))['grid']['read']['true']['whole']
    keep = ans >= MIN_ANSWERS
    print(f'{tag}: answers whole with the true address {ans:.4f} -> '
          + ('its members train' if keep else f'NOT RUN (under {MIN_ANSWERS}: no member can answer on it)'))
    if keep:
        CORES.append((tag, seed, forms))
assert CORES, 'no core answers with the true address: no member trains'
CORE_DIRS = {}
for tag, seed, forms in CORES:
    snapshot_download(REPO, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/hub', token=HF_TOKEN or False)
    CORE_DIRS[tag] = f'{OUT}/hub/bed/weights/{tag}'
    assert all(os.path.exists(f'{CORE_DIRS[tag]}/{f}') for f in ('config.json', 'model.safetensors', 'scale.json', 'addr_head.safetensors')), tag
print('cores:', [(t, f) for t, _, f in CORES])
owned, never = rs.owned_sets()
packed = rs.load_packed(BED_DIR)
core = rs.Core(CORE_DIRS[CORES[-1][0]], 'cuda')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
cc = rs.compile_check(core, packed, owned, never)
print('compiled = eager at step 1 (largest difference over the largest value):', cc)
assert all(v['write'] < 1e-4 and v['grad'] < 1e-3 for v in cc.values()), 'the compiled writes are not the eager writes: no member trains'
pr = rs.price(core, packed, owned, never, steps=PRICE_STEPS)
per_core = (pr['last_ms_per_step'] * LAST_STEPS + pr['mid_ms_per_step'] * MID_STEPS) / 60000
PRICE_OK = per_core <= STOP_FACTOR * PLAN_MIN
print(f"this card: last site {pr['last_ms_per_step']:.1f} ms a step, mid site {pr['mid_ms_per_step']:.1f} ms a step -> a core's members about "
      f"{per_core:.0f} min of training (planned {PLAN_MIN}) + the reads; the {len(CORES)} cores about {per_core * len(CORES) / 60:.1f} h + reads"
      + ('' if PRICE_OK else f' -> PAST {STOP_FACTOR}x THE PLAN: C3 will not start; re-plan first'))
del core
torch.cuda.empty_cache()
PUSH_OK = False
if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    path = 'bed/preflight/push_probe_find_members.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                       'compile_check': cc, 'price': pr, 'cores': [t for t, _, _ in CORES]}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano find members: push probe')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; C3 runs and ships nothing')

In [ ]:
# C3 (T) — THE SESSION: the cores in order, each core's last site (train, read, text bits) then its mid site (train, read), resume-first;
# a STOP file in OUT starts no new core; every finished core's folder ships in the background (the card never waits on the Hub).
assert PRICE_OK, 'the price line projected past the plan: re-plan before any member trains'
shipper = rs.Shipper(REPO) if (HF_TOKEN and PUSH_OK) else None
done = []
t_all = time.time()
for n, (tag, seed, forms) in enumerate(CORES, 1):
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new core starts')
        break
    print(f'=================== {n}/{len(CORES)} {tag} (read forms {forms}) ===================')
    t0 = time.time()
    res = rs.run_core(CORE_DIRS[tag], f'{OUT}/members', seed, BED_DIR, HELD, last_steps=LAST_STEPS, mid_steps=MID_STEPS,
                      read_batches=READ_BATCHES, calib_batches=CALIB_BATCHES, shipper=shipper, forms=forms)
    done.append(res)
    torch.cuda.empty_cache()
    print(f'{tag}: {(time.time() - t0) / 60:.1f} min; {(time.time() - t_all) / 60:.1f} min spent; {len(CORES) - n} cores left')
if shipper:
    print('ships drained' if shipper.drain() else 'ships still retrying in the background')
print('session:', [r['name'] for r in done])

In [ ]:
# C4 — THE TABLE: per core and stack, solo against the bare core, the nine rest cells keeping all three marks for each request source (the
# two-pass form), the controls, the largest leak of the claim's sources, the mid site's claim cells, the text bits. Run any time.
import glob
from IPython.display import Markdown, display
rows = ['| core | stack | solo / bare owned | HEADIF | HEADIFQ999 | INDEXF | LAYOUT | G2 m48 | INDEXC m48 Q | G0 m48 P/Q | largest leak / own (HEADIF) | mid HEADIF U5/8/10 | G0 text bits |',
        '|---|---|---|---|---|---|---|---|---|---|---|---|---|']
for d in sorted(glob.glob(f'{OUT}/members/*_members_*')):
    if not os.path.exists(f'{d}/last_find.json'):
        continue
    L = json.load(open(f'{d}/last_find.json', encoding='utf-8'))
    M = json.load(open(f'{d}/mid_find.json', encoding='utf-8')) if os.path.exists(f'{d}/mid_find.json') else None
    rd = L['read']
    for st, r in rd['stacks'].items():
        c = r['cells']
        allm = lambda x: all(x[k] for k in ('PROTECTED', 'QUIET', 'EXACT'))
        nine = lambda s: f"{sum(allm(c[f'{s}_U{u}_REST_m{m}']) for u in (5, 8, 10) for m in (16, 32, 48))}/9" if f'{s}_U5_REST_m16' in c else '-'
        leak = (max(c[f'HEADIF_U{u}_REST_m{m}']['leak_over_own_write'] for u in (5, 8, 10) for m in (16, 32, 48))
                if 'HEADIF_U5_REST_m16' in c else float('nan'))
        mid = '-'
        if M and st == 'arms':
            mc = M['read']['cells']
            mid = ' '.join(str(allm(mc[f'HEADIF_U{u}_REST_m16'])) if f'HEADIF_U{u}_REST_m16' in mc else '-' for u in (5, 8, 10))
        g0 = L['text_bits'].get(f'G0_{st}_minus_bare')
        rows.append(f"| {L['core'][:26]} | {st} | {r['solo']['ONE']['48']:.3f} / {rd['bare']['owned']:.3f} | {nine('HEADIF')} | {nine('HEADIFQ999')} | "
                    f"{nine('INDEXF')} | {nine('LAYOUT')} | {allm(c['G2_m48'])} | {c['INDEXC_U5_REST_m48']['QUIET']} | "
                    f"{c['G0_m48']['PROTECTED']}/{c['G0_m48']['QUIET']} | {leak:.1e} | {mid} | {g0:+.3f} |")
display(Markdown('\n'.join(rows)))